# Day 4(M1 Day04) 확장 실습 — A2A로 다른 팀 에이전트에게 일 맡기기

**목표**: 본 실습에서 MCP로 **자료**를 다른 서버에서 받아 왔다. 이번에는 **분석가 한 명**을 다른 팀의 서버로 떼어 내고, 에이전트끼리 일을 주고받는 규격 **A2A**로 일을 맡긴다. 상대를 찾고(에이전트 카드) → 할 줄 아는지 확인하고 → 맡겨서 결과를 받아, 본 실습의 통합 보고서에 합친다.
## 오늘의 전체 그림

**오늘의 질문**: 다른 팀이 만든 에이전트에게 일을 맡기려면, 무엇을 알아야 하고 어떻게 부탁하나? 상대가 응답하지 못하면?

```text
[Part 1 서버]    다른 팀의 고객 분석가 — 에이전트 카드 AGENT_CARD · SendMessage 처리 (FastAPI)
      │           → 카드 주소와 JSON-RPC 주소
      ▼
[Part 2 클라이언트] 찾기(카드 읽기) → 확인 find_endpoint → 맡기기 send_message
      │           → 규격 이름이 다르면 거절된다 (옛 이름 message/send)
      ▼
[Part 3 팀에 붙이기] 우리 팀 분석가 2명 + 다른 팀 분석가 1명(A2A) → 통합자
                  → 다른 팀 서버가 고장 나면 빠진 관점을 표시하고 계속
```

| Part | 쪽 | 답하는 질문 | 다음 Part로 넘기는 것 |
| --- | --- | --- | --- |
| Part 1 | 서버 | 내 에이전트를 다른 팀이 쓰게 하려면 무엇을 내놓나? | `AGENT_CARD` · `customer_team` · `network` |
| Part 2 | 클라이언트 | 다른 팀 에이전트를 어떻게 찾고 확인하고 맡기나? | `find_endpoint` · `send_message` |
| Part 3 | 팀 | 맡긴 결과를 우리 팀 결과와 어떻게 합치나? | 통합 보고서 |

> 💡 세 Part는 코드를 이어 쓴다. 앞 Part의 셀을 실행하지 않으면 뒤 Part가 돌지 않는다.

**선수**: 본 실습 Part 2(통합자), 4절 "컨텍스트 전달 네 가지"(배포용 문서). 1과목의 FastAPI.

> 💡 **규격 버전**: A2A **1.0.0** 기준이다 (2026-10-11 공식 문서 a2a-protocol.org와 규격 정의 파일로 확인). 책 『AI 에이전트 엔지니어링』 8.5.2의 예제는 그 전 버전이라 이름이 다르다 — Part 2에서 직접 부딪쳐 본다.

> ⚠️ **최소 구현이다.** 규격 중 JSON-RPC로 `SendMessage` 하나만, 인증 없이 손으로 만든다. 실제 서비스는 인증·스트리밍·작업 조회까지 갖춰야 한다. 서버는 진짜 네트워크 대신 FastAPI `TestClient`로 같은 노트북 안에서 부른다. 이 노트북은 LLM을 6~8번 부른다.

## 0. 환경 준비

본 실습과 같은 자료(달빛 도서관)를 쓴다. 새로 쓰는 것은 FastAPI의 `TestClient` 하나다 — 서버를 따로 띄우지 않고 같은 프로그램 안에서 서버의 주소를 부른다.

> 💡 실행하면 `StarletteDeprecationWarning`(httpx 대신 httpx2를 쓰라는 안내) 경고가 한 번 나올 수 있다. `TestClient`가 안에서 쓰는 라이브러리에 대한 안내이고 실습과 무관하다 (fastapi 0.139.0, 2026-10-11 확인).

In [ ]:
import os

# 추적 끄기: 2과목 Day17에서 .env의 LANGSMITH_TRACING을 켰다면 오늘 호출이 전부 전송된다
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

import uuid

from dotenv import load_dotenv
from fastapi import FastAPI
from fastapi.testclient import TestClient
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SOURCE = """[내부 소개 자료] 달빛 도서관
달빛 도서관은 밤 10시부터 새벽 2시까지 여는 무인 야간 도서관이다.
서울 마포구에 1호점이 있으며, 좌석은 총 48석이다.
입장은 월 구독제이며, 구독료는 월 19,900원이다.
사서 대신 안내 로봇 '루나'가 책 추천과 반납을 돕는다.
조용한 환경을 위해 통화와 취식은 금지된다.
회원의 62%가 20~30대 직장인이다.
2026년 12월에 부산 2호점이 문을 열 예정이다."""   # 본 실습과 같은 자료

print("준비 완료:", llm.model_name)

---

# Part 1. 서버 — 다른 팀의 고객 분석가

## 1-1. 에이전트 카드 — "나는 누구고 무엇을 할 줄 안다"

다른 팀이 만든 **고객 분석가**가 있다고 하자. 다른 팀이 이 에이전트를 쓰려면 먼저 알아야 할 것이 있다. 그것을 적어 정해진 주소(`/.well-known/agent-card.json`)에 걸어 두는 JSON이 **에이전트 카드**다.

| 칸 (규격 1.0.0) | 뜻 |
| --- | --- |
| `name`, `description`, `version` | 이름, 설명, 이 에이전트의 버전 |
| `supportedInterfaces` | 어디로 어떤 방식으로 부르나 — `url`, `protocolBinding`(여기서는 `JSONRPC`), `protocolVersion` |
| `skills` | 할 줄 아는 일 목록 — `id`, `name`, `description`, `tags` |
| `capabilities` | 스트리밍 같은 부가 기능을 지원하나 |
| `defaultInputModes`, `defaultOutputModes` | 주고받는 내용 형식 (여기서는 글 `text/plain`) |

MCP 서버가 "제공하는 도구 목록"을 알려 주는 것과 같은 역할이다. 인증 방식도 카드에 적지만, 오늘은 최소 구현이라 뺐다.

In [ ]:
AGENT_CARD = {
    "name": "고객 분석가 (고객경험팀)",
    "description": "서비스 소개 자료를 이용자 입장에서 읽고 좋은 점과 불편한 점을 찾는다.",
    "version": "1.0.0",
    "supportedInterfaces": [
        # testserver: TestClient가 쓰는 가짜 서버 주소. 실제로는 그 팀 서버의 주소가 들어간다
        {"url": "http://testserver/a2a", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"},
    ],
    "capabilities": {"streaming": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {"id": "customer-analysis", "name": "고객 관점 분석",
         "description": "자료를 이용자 입장에서 읽고 좋은 점 2개, 불편한 점 2개를 찾는다.",
         "tags": ["분석", "고객"]},
    ],
}
print("기능:", [skill["id"] for skill in AGENT_CARD["skills"]])

## 1-2. 일을 받는 쪽 — `SendMessage` 처리

다른 팀은 JSON-RPC 형식으로 일을 보낸다. JSON-RPC는 "이 이름의 기능(`method`)을 이 값(`params`)으로 실행해 달라"는 JSON이다 (MCP도 같은 형식을 쓴다).

| 받는 요청 | 돌려주는 응답 |
| --- | --- |
| `method`: `"SendMessage"` | 성공하면 `result` 안에 `task` — 작업 번호, 상태 `TASK_STATE_COMPLETED`, 결과물 `artifacts` |
| `params.message`: 메시지 번호, 역할 `ROLE_USER`, 내용 `parts` | 모르는 메서드면 `error` — 코드 `-32601` (Method not found) |

서버는 FastAPI로 만든다. 주소 두 개 — 카드를 보여 주는 `GET`, 일을 받는 `POST` — 면 된다.

In [ ]:
def analyze_customer(text):
    """다른 팀의 고객 분석가가 실제로 하는 일 (본 실습의 고객 분석가와 같은 임무)."""
    res = llm.invoke([("system", "당신은 고객 분석가입니다. 자료를 이용자 입장에서 읽고 "
                                 "좋은 점 2개, 불편한 점 2개를 한 줄씩 쓰세요. 자료에 있는 내용만 근거로 씁니다."),
                      ("human", text)])
    return res.content.strip()


def rpc_error(request_id, code, message):
    return {"jsonrpc": "2.0", "id": request_id, "error": {"code": code, "message": message}}


customer_team = FastAPI()


@customer_team.get("/.well-known/agent-card.json")
def get_agent_card():
    return AGENT_CARD


@customer_team.post("/a2a")
def handle_rpc(body: dict):
    request_id = body.get("id")
    # TODO: 규격 1.0.0에서 메시지를 보내는 메서드 이름을 문자열로 넣으세요
    if body.get("method") != ____:
        return rpc_error(request_id, -32601, "Method not found")
    message = body["params"]["message"]
    text = "".join(part["text"] for part in message["parts"] if "text" in part)
    task = {"id": str(uuid.uuid4()), "contextId": str(uuid.uuid4()),
            "status": {"state": "TASK_STATE_COMPLETED"},
            "artifacts": [{"artifactId": str(uuid.uuid4()), "parts": [{"text": analyze_customer(text)}]}]}
    return {"jsonrpc": "2.0", "id": request_id, "result": {"task": task}}


network = TestClient(customer_team)   # 진짜 네트워크 대신, 같은 프로그램 안에서 서버를 부른다
print("다른 팀 서버 준비 완료")

**관찰** (실측): 서버 준비가 끝났다. 서버가 하는 일은 두 가지뿐이다 — 카드를 보여 주고(`GET`), 정해진 이름의 요청(`SendMessage`)만 받아 분석을 돌려준다(`POST`). 고객 분석가가 안에서 무엇으로 돌아가는지(어떤 모델, 어떤 프롬프트)는 카드에 없다. 맡기는 쪽은 **무엇을 할 줄 아는지**만 알면 된다.

---

# Part 2. 클라이언트 — 찾고, 확인하고, 맡기기

## 2-1. 찾기 — 카드 읽기

이제 우리 팀 쪽이다. 다른 팀 서버의 정해진 주소에서 카드를 읽는다.

In [ ]:
card = network.get("/.well-known/agent-card.json").json()
print("이름:", card["name"])
print("할 줄 아는 일:", [skill["id"] for skill in card["skills"]])
print("접속 정보:", card["supportedInterfaces"][0])

**관찰** (실측): 카드에서 이름, 기능 하나(`customer-analysis`), 접속 정보(주소 `http://testserver/a2a`, 방식 `JSONRPC`, 규격 `1.0`)를 읽었다. 아직 아무 일도 맡기지 않았다 — 찾기는 카드만 읽는다.

## 2-2. 확인 — 필요한 기능이 있나, 같은 방식으로 말하나

카드를 읽었다고 바로 맡기지 않는다. 두 가지를 먼저 본다.

| 확인 | 없으면 |
| --- | --- |
| 필요한 기능(`skills`의 `id`)이 있나 | 할 줄 모르는 일을 맡기게 된다 |
| 우리가 쓰는 방식(JSON-RPC, 규격 1.x)으로 받나 | 보내도 알아듣지 못한다 |

In [ ]:
def find_endpoint(card, skill_id):
    """카드에서 기능과 접속 방식을 확인한다. 맞으면 JSON-RPC 주소, 아니면 None."""
    skill_ids = [skill["id"] for skill in card["skills"]]
    # TODO: 필요한 기능 id가 카드의 기능 목록에 없는지 확인하세요
    if skill_id not in ____:
        print(f"  [확인 실패] '{skill_id}' 기능이 없다. 가진 기능: {skill_ids}")
        return None
    for interface in card["supportedInterfaces"]:
        if interface["protocolBinding"] == "JSONRPC" and interface["protocolVersion"].startswith("1."):
            return interface["url"]
    print("  [확인 실패] JSON-RPC 규격 1.x로 받는 주소가 없다")
    return None


endpoint = find_endpoint(card, "customer-analysis")
print("고객 분석 →", endpoint)
print("번역 →", find_endpoint(card, "translation"))

**관찰** (실측): 고객 분석은 주소를 돌려받았고, 카드에 없는 `translation`은 `[확인 실패]`와 함께 `None`이었다. 할 줄 모르는 일은 **보내기 전에** 걸러진다.

## 2-3. 맡기기 — `SendMessage`

메시지 하나를 만들어 보낸다. 메시지에는 번호(`messageId`), 보내는 쪽 역할(`ROLE_USER`), 내용(`parts`)이 들어간다. 응답의 `task`에서 상태와 결과물을 꺼낸다.

응답을 꺼내기 전에 두 가지를 먼저 본다 — 서버가 아예 응답하지 못했나(HTTP 상태), 응답은 했는데 거절했나(`error`). 다른 서버에 맡기는 일이라 둘 다 일어날 수 있다 (배포용 문서 5-2 분산 통신).

In [ ]:
def send_message(url, text, method="SendMessage", client=network):
    """A2A로 일을 맡긴다. (작업 상태, 결과 글)을 돌려준다."""
    request = {"jsonrpc": "2.0", "id": str(uuid.uuid4()), "method": method,
               "params": {"message": {"messageId": str(uuid.uuid4()), "role": "ROLE_USER",
                                      "parts": [{"text": text}]}}}
    response = client.post(url, json=request)
    if response.status_code != 200:
        return "UNAVAILABLE", f"HTTP {response.status_code}"          # 서버가 응답하지 못했다
    reply = response.json()
    if "error" in reply:
        return "ERROR", f"{reply['error']['code']} {reply['error']['message']}"   # 거절했다
    # TODO: 응답(reply)의 result 안에서 작업(task)을 꺼내세요
    task = ____
    result_text = ""
    for artifact in task.get("artifacts", []):
        for part in artifact["parts"]:
            result_text += part.get("text", "")
    return task["status"]["state"], result_text


state, answer = send_message(endpoint, SOURCE)
print("상태:", state)
print(answer)

**관찰** (실측): 상태 `TASK_STATE_COMPLETED`와 함께 좋은 점 2개·불편한 점 2개가 돌아왔다. 결과는 작업(`task`) 안의 결과물(`artifacts`) 안의 내용(`parts`)에 들어 있다 — 그래서 꺼내는 코드가 두 겹 반복문이다.

## 2-4. 옛 이름으로 보내면

책 『AI 에이전트 엔지니어링』 8.5.2의 예제는 메시지를 `message/send`라는 이름으로 보낸다. 그 사이 규격이 1.0.0으로 바뀌며 이름이 `SendMessage`가 됐다. 옛 이름으로 보내면 어떻게 될까?

In [ ]:
state, answer = send_message(endpoint, SOURCE, method="message/send")
print("상태:", state)
print(answer)

**관찰** (실측): 상태 `ERROR`, `-32601 Method not found`. 1.0.0 서버는 옛 이름을 모른다. 책의 예제를 그대로 따라 하면 지금 규격의 서버와는 말이 통하지 않는다는 뜻이다. 2-2의 확인에서 규격 버전(`protocolVersion`)을 본 이유이고, 규격이 바뀌는 기술은 **책보다 공식 문서의 최신 버전**을 먼저 본다.

---

# Part 3. 팀에 붙이기 — 우리 팀 2명 + 다른 팀 1명

## 3-1. 세 관점 모으기

본 실습의 분석 팀은 세 관점(낙관·비관·고객)을 한 노트북 안에서 돌렸다. 이번에는 낙관·비관은 **우리 팀**이, 고객은 **다른 팀**(A2A)이 맡는다. 다른 팀이 일을 끝내지 못하면 그 관점은 "받지 못함"으로 남긴다.

In [ ]:
LOCAL_PERSPECTIVES = {
    "낙관": "이 서비스의 강점과 성장 기회를 찾는다",
    "비관": "이 서비스의 약점과 사업 위험을 찾는다",
}


def analyze_local(name, mission, text):
    """우리 팀 분석가 (본 실습의 분석가를 한 줄 글로 줄인 것)."""
    res = llm.invoke([("system", f"당신은 {name} 분석가입니다. 임무: {mission}. "
                                 "자료에 있는 내용만 근거로 2문장으로 쓰세요."),
                      ("human", text)])
    return res.content.strip()


def collect_analyses(text, client=network):
    analyses = []
    for name, mission in LOCAL_PERSPECTIVES.items():
        analyses.append({"관점": name, "맡은 곳": "우리 팀", "분석": analyze_local(name, mission, text), "받음": True})
    # 다른 팀에는 A2A로 맡긴다
    # TODO: 2-3의 함수로 다른 팀에 고객 분석을 맡기세요 (주소 endpoint, 자료 text, client=client)
    state, remote = ____
    if state == "TASK_STATE_COMPLETED":
        analyses.append({"관점": "고객", "맡은 곳": "다른 팀 (A2A)", "분석": remote, "받음": True})
    else:
        analyses.append({"관점": "고객", "맡은 곳": "다른 팀 (A2A)", "분석": f"(받지 못함: {state} {remote})",
                         "받음": False})
    return analyses


analyses = collect_analyses(SOURCE)
for a in analyses:
    print(f"[{a['관점']} · {a['맡은 곳']}] {a['분석'][:80]}")

**관찰** (실측): 세 관점이 모였다. 낙관·비관은 이 노트북 안에서, 고객은 다른 팀 서버에서 왔다. 모은 목록에는 관점마다 `맡은 곳`과 `받음`이 함께 적혀 있다.

## 3-2. 통합 — 통합자는 어디서 왔는지 모른다

본 실습의 통합 원칙대로, 관점이 부딪치는 점은 뭉개지 않고 남긴다.

In [ ]:
def aggregate(analyses):
    """관점별 분석을 종합한다. 받지 못한 관점은 빠졌다고 밝힌다."""
    lines = [f"[{a['관점']}] {a['분석']}" for a in analyses]
    res = llm.invoke([("system", "당신은 통합자입니다. 관점별 분석을 3문장으로 종합하세요. "
                                 "관점이 부딪치는 점이 있으면 마지막에 '충돌:'로 한 줄 남기고, "
                                 "'받지 못함'인 관점이 있으면 '빠진 관점:'으로 한 줄 남기세요."),
                      ("human", "\n".join(lines))])
    return res.content.strip()


print(aggregate(analyses))

**관찰** (실측): 종합과 충돌은 잘 남겼다. 그런데 마지막 줄에 **"빠진 관점: 경영적 관점"** 이 붙었다 — 세 관점을 모두 받았는데 통합자가 없는 빈칸을 지어낸 것이다 (자료를 만들며 두 번 실행해 두 번 모두 빠진 관점을 지어냈다). "받지 못한 관점이 있으면 적으라"는 지시를 "빠진 관점을 찾아 적으라"로 받아들였다.

## 3-2b. 빠진 관점은 코드가 적는다

"어느 관점을 받지 못했나"는 판단이 아니라 **사실**이다. 3-1의 `collect_analyses`가 관점마다 `받음` 칸에 이미 적어 두었다. 이런 것은 LLM에게 묻지 않고 코드가 적는다 (Day 1 "판정은 코드로"). 통합자에게는 받은 분석만 주고 종합만 맡긴다.

In [ ]:
def aggregate_v2(analyses):
    """종합은 LLM이, 빠진 관점은 코드가 적는다."""
    received = [a for a in analyses if a["받음"]]
    # TODO: 받지 못한(받음이 False인) 관점의 이름만 모으세요
    missing = ____
    lines = [f"[{a['관점']}] {a['분석']}" for a in received]
    res = llm.invoke([("system", "당신은 통합자입니다. 관점별 분석을 3문장으로 종합하세요. "
                                 "관점이 부딪치는 점이 있으면 마지막에 '충돌:'로 한 줄 남기세요."),
                      ("human", "\n".join(lines))])
    report = res.content.strip()
    if missing:
        report += "\n빠진 관점: " + ", ".join(missing) + " (다른 팀에서 받지 못함)"
    return report


print(aggregate_v2(analyses))

**관찰** (실측): 이번에는 빠진 관점 줄이 없다 — 받지 못한 관점이 없으니 코드가 아무것도 덧붙이지 않았다. 통합자는 종합과 충돌만 맡았다. 무엇을 받았는지는 코드가 이미 아는 **사실**이라, LLM에게 판단을 맡기면 3-2처럼 지어낼 틈만 생긴다.

## 3-3. 다른 팀 서버가 고장 나면

다른 서버에 맡긴 일은 우리가 고칠 수 없는 이유로 실패한다. 다른 팀 서버가 오류를 내는 상황을 만들어, 팀 전체가 멈추는지 본다.

In [ ]:
broken_team = FastAPI()


@broken_team.post("/a2a")
def handle_rpc_broken(body: dict):
    raise RuntimeError("다른 팀 서버 장애")   # 일부러 만든 장애


# raise_server_exceptions=False: 서버 예외를 파이썬 오류로 멈추지 않고 500 응답으로 받는다 (실제 서버와 같다)
broken_network = TestClient(broken_team, raise_server_exceptions=False)

analyses_broken = collect_analyses(SOURCE, client=broken_network)
for a in analyses_broken:
    print(f"[{a['관점']} · {a['맡은 곳']}] {a['분석'][:60]}")
print()
print(aggregate_v2(analyses_broken))

**관찰** (실측): 다른 팀 서버가 500을 냈지만 팀은 멈추지 않았다. `send_message`가 HTTP 상태를 먼저 보고 `UNAVAILABLE`을 돌려줬고, 고객 관점은 `(받지 못함: UNAVAILABLE HTTP 500)`으로 남았다. 보고서는 받은 두 관점으로 쓰였고, 마지막 줄에 **"빠진 관점: 고객 (다른 팀에서 받지 못함)"** 이 코드로 정확히 붙었다. 다른 서버에 맡기는 일은 언제든 실패할 수 있다 — 실패를 **숨기지 않고 표시한 채** 계속하는 것이 분산 통신의 기본이다 (자동 재시도·대체 경로는 M2 Day 10).

---

## 확인 문제

1. MCP와 A2A는 각각 무엇과 무엇을 잇는가? 오늘 실습에서 각각 무엇에 썼다면?
2. 에이전트 카드에 반드시 있어야 하는 것 두 가지와, 클라이언트가 그것으로 확인한 것은?
3. 2-4에서 옛 이름 `message/send`로 보내자 무슨 일이 일어났나? 그것이 알려 주는 것은?
4. `send_message`가 응답을 꺼내기 전에 HTTP 상태와 `error`를 먼저 보는 이유는?
5. 3-2에서 통합자에게 빠진 관점까지 적게 했더니 무슨 일이 있었나? 3-2b는 무엇을 바꿨나?
6. 3-3에서 다른 팀 서버가 고장 났을 때 팀은 어떻게 됐나?
7. 이번 확장은 서버를 진짜 네트워크가 아니라 `TestClient`로 불렀다. 실제로 다른 팀 서버를 부르면 무엇을 더 신경 써야 하나?

## 정리·회고

| 본 실습 (MCP) | 이번 확장 (A2A) |
| --- | --- |
| 다른 서버의 **자료**를 받는다 | 다른 팀의 **에이전트**에게 일을 맡긴다 |
| 도구 목록을 보고 도구를 고른다 | 에이전트 카드를 보고 기능·접속 방식을 확인한다 |
| `.py` 스크립트로 서버를 띄운다 (stdio) | HTTP로 부른다 (오늘은 `TestClient`) |
| 받은 자료를 분석가가 쓴다 | 받은 분석을 통합자가 합친다 |

| A2A 3단계 | 오늘 코드 | 빠지면 |
| --- | --- | --- |
| 찾기 | `/.well-known/agent-card.json` 읽기 | 상대가 무엇을 하는지 모른다 |
| 확인 | `find_endpoint` | 할 줄 모르는 일을 맡기거나, 말이 안 통한다 (2-4) |
| 맡기기 | `send_message` (`SendMessage`) | — |

1. 우리 팀 서비스(상품 소개문 서비스)를 다른 팀이 A2A로 쓰게 하려면 카드의 `skills`에 무엇을 적겠는가?
2. 규격이 바뀌는 중인 기술을 수업·업무에 쓸 때, 책과 공식 문서 중 무엇을 먼저 보는가? 왜인가?